<a href="https://colab.research.google.com/github/MacuxiDev/sistema_copa_2026/blob/main/SistemaCopa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install sqlalchemy pydantic requests --quiet

In [3]:
import sqlalchemy
import pydantic
print(f"SQLAlchemy: {sqlalchemy.__version__}")
print(f"Pydantic: {pydantic.__version__}")

SQLAlchemy: 2.0.51
Pydantic: 2.13.4


In [4]:
# Célula 2 — Configuração
from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, Session

# Banco local no Colab
DATABASE_URL = "sqlite:///apostas_copa2026.db"
engine = create_engine(DATABASE_URL, echo=True)

class Base(DeclarativeBase):
    pass

print("✅ Engine configurada com sucesso")

✅ Engine configurada com sucesso


In [5]:
# Célula 3 — Models SQLAlchemy
from sqlalchemy import (
    Column, Integer, String, Float,
    DateTime, ForeignKey, CheckConstraint,
    UniqueConstraint, Text
)
from sqlalchemy.orm import relationship
from datetime import datetime, timezone

# ─────────────────────────────────────
# 1. FASE
# ─────────────────────────────────────
class Fase(Base):
    __tablename__ = "fase"

    id_fase  = Column(Integer, primary_key=True, autoincrement=True)
    nome     = Column(String(50), nullable=False, unique=True)
    ordem    = Column(Integer, nullable=False, unique=True)
    tipo     = Column(String(20), nullable=False)

    __table_args__ = (
        CheckConstraint("tipo IN ('grupos','eliminatoria')", name="ck_fase_tipo"),
        CheckConstraint("ordem > 0", name="ck_fase_ordem"),
    )

    grupos   = relationship("Grupo",   back_populates="fase")
    partidas = relationship("Partida", back_populates="fase")

    def __repr__(self):
        return f"<Fase {self.nome}>"


# ─────────────────────────────────────
# 2. GRUPO
# ─────────────────────────────────────
class Grupo(Base):
    __tablename__ = "grupo"

    id_grupo  = Column(Integer, primary_key=True, autoincrement=True)
    id_fase   = Column(Integer, ForeignKey("fase.id_fase"), nullable=False)
    nome      = Column(String(10), nullable=False)
    descricao = Column(String(100))

    __table_args__ = (
        UniqueConstraint("id_fase", "nome", name="uq_grupo_fase_nome"),
    )

    fase     = relationship("Fase",    back_populates="grupos")
    selecoes = relationship("Selecao", back_populates="grupo")

    def __repr__(self):
        return f"<Grupo {self.nome}>"


# ─────────────────────────────────────
# 3. SELECAO
# ─────────────────────────────────────
class Selecao(Base):
    __tablename__ = "selecao"

    id_selecao = Column(Integer, primary_key=True, autoincrement=True)
    id_grupo   = Column(Integer, ForeignKey("grupo.id_grupo"), nullable=False)
    nome       = Column(String(60), nullable=False)
    sigla      = Column(String(3),  nullable=False, unique=True)

    grupo = relationship("Grupo", back_populates="selecoes")

    partidas_a = relationship(
        "Partida",
        foreign_keys="Partida.id_selecao_a",
        back_populates="selecao_a"
    )
    partidas_b = relationship(
        "Partida",
        foreign_keys="Partida.id_selecao_b",
        back_populates="selecao_b"
    )

    def __repr__(self):
        return f"<Selecao {self.sigla}>"


# ─────────────────────────────────────
# 4. PARTIDA
# ─────────────────────────────────────
class Partida(Base):
    __tablename__ = "partida"

    id_partida   = Column(Integer, primary_key=True, autoincrement=True)
    id_fase      = Column(Integer, ForeignKey("fase.id_fase"),       nullable=False)
    id_selecao_a = Column(Integer, ForeignKey("selecao.id_selecao"), nullable=False)
    id_selecao_b = Column(Integer, ForeignKey("selecao.id_selecao"), nullable=False)
    num_jogo     = Column(Integer, nullable=False, unique=True)
    data_hora    = Column(DateTime)
    status       = Column(String(20), nullable=False, default="agendada")
    gols_a       = Column(Integer, default=0)
    gols_b       = Column(Integer, default=0)
    odd_a        = Column(Float, nullable=False)
    odd_b        = Column(Float, nullable=False)
    odd_empate   = Column(Float, nullable=False)

    __table_args__ = (
        CheckConstraint("id_selecao_a != id_selecao_b",           name="ck_partida_selecoes_diferentes"),
        CheckConstraint("status IN ('agendada','ao_vivo','encerrada','cancelada')", name="ck_partida_status"),
        CheckConstraint("odd_a > 0",      name="ck_partida_odd_a"),
        CheckConstraint("odd_b > 0",      name="ck_partida_odd_b"),
        CheckConstraint("odd_empate > 0", name="ck_partida_odd_empate"),
        CheckConstraint("gols_a >= 0",    name="ck_partida_gols_a"),
        CheckConstraint("gols_b >= 0",    name="ck_partida_gols_b"),
    )

    fase      = relationship("Fase",    back_populates="partidas")
    selecao_a = relationship("Selecao", foreign_keys=[id_selecao_a], back_populates="partidas_a")
    selecao_b = relationship("Selecao", foreign_keys=[id_selecao_b], back_populates="partidas_b")
    apostas   = relationship("Aposta",  back_populates="partida")

    def __repr__(self):
        return f"<Partida {self.num_jogo}>"


# ─────────────────────────────────────
# 5. USUARIO
# ─────────────────────────────────────
class Usuario(Base):
    __tablename__ = "usuario"

    id_usuario      = Column(Integer, primary_key=True, autoincrement=True)
    nome            = Column(String(100), nullable=False)
    email           = Column(String(120), nullable=False, unique=True)
    cpf             = Column(String(14),  nullable=False, unique=True)
    data_nascimento = Column(DateTime,    nullable=False)
    login           = Column(String(50),  nullable=False, unique=True)
    senha_hash      = Column(String(255), nullable=False)
    status          = Column(String(20),  nullable=False, default="ativo")
    tipo_usuario    = Column(String(20),  nullable=False, default="usuario")

    __table_args__ = (
        CheckConstraint("status IN ('ativo','inativo','bloqueado')",             name="ck_usuario_status"),
        CheckConstraint("tipo_usuario IN ('usuario','administrador')",           name="ck_usuario_tipo"),
    )

    conta   = relationship("Conta_Pontos", back_populates="usuario", uselist=False)
    apostas = relationship("Aposta",       back_populates="usuario")

    def __repr__(self):
        return f"<Usuario {self.login}>"


# ─────────────────────────────────────
# 6. CONTA_PONTOS
# ─────────────────────────────────────
class Conta_Pontos(Base):
    __tablename__ = "conta_pontos"

    id_conta        = Column(Integer, primary_key=True, autoincrement=True)
    id_usuario      = Column(Integer, ForeignKey("usuario.id_usuario"), nullable=False, unique=True)
    saldo           = Column(Float,   nullable=False, default=100.0)
    data_atualizacao = Column(DateTime, default=lambda: datetime.now(timezone.utc))

    __table_args__ = (
        CheckConstraint("saldo >= 0", name="ck_conta_saldo"),
    )

    usuario        = relationship("Usuario",              back_populates="conta")
    movimentacoes  = relationship("Movimentacao_Pontos",  back_populates="conta")

    def __repr__(self):
        return f"<Conta usuario={self.id_usuario} saldo={self.saldo}>"


# ─────────────────────────────────────
# 7. APOSTA
# ─────────────────────────────────────
class Aposta(Base):
    __tablename__ = "aposta"

    id_aposta        = Column(Integer, primary_key=True, autoincrement=True)
    id_usuario       = Column(Integer, ForeignKey("usuario.id_usuario"), nullable=False)
    id_partida       = Column(Integer, ForeignKey("partida.id_partida"), nullable=False)
    palpite          = Column(String(10), nullable=False)
    pontos_apostados = Column(Float,      nullable=False)
    multiplicador    = Column(Integer,    nullable=False, default=1)
    odd_aplicada     = Column(Float,      nullable=False)
    retorno_potencial = Column(Float,     nullable=False)
    status           = Column(String(20), nullable=False, default="ativa")
    data_hora        = Column(DateTime,   default=lambda: datetime.now(timezone.utc))

    __table_args__ = (
        CheckConstraint("palpite IN ('selecao_a','empate','selecao_b')", name="ck_aposta_palpite"),
        CheckConstraint("pontos_apostados > 0",                          name="ck_aposta_pontos"),
        CheckConstraint("multiplicador IN (1,2,3,4,5)",                  name="ck_aposta_multiplicador"),
        CheckConstraint("odd_aplicada > 0",                              name="ck_aposta_odd"),
        CheckConstraint("status IN ('ativa','ganha','perdida','reembolsada','anulada')", name="ck_aposta_status"),
    )

    usuario  = relationship("Usuario",  back_populates="apostas")
    partida  = relationship("Partida",  back_populates="apostas")
    movimentacoes = relationship("Movimentacao_Pontos", back_populates="aposta")

    def __repr__(self):
        return f"<Aposta {self.id_aposta} | {self.palpite} | {self.status}>"


# ─────────────────────────────────────
# 8. MOVIMENTACAO_PONTOS
# ─────────────────────────────────────
class Movimentacao_Pontos(Base):
    __tablename__ = "movimentacao_pontos"

    id_movimentacao   = Column(Integer, primary_key=True, autoincrement=True)
    id_conta          = Column(Integer, ForeignKey("conta_pontos.id_conta"), nullable=False)
    id_aposta         = Column(Integer, ForeignKey("aposta.id_aposta"),      nullable=True)
    tipo_movimentacao = Column(String(20), nullable=False)
    pontos            = Column(Float,      nullable=False)
    saldo_anterior    = Column(Float,      nullable=False)
    saldo_atual       = Column(Float,      nullable=False)
    data_hora         = Column(DateTime,   default=lambda: datetime.now(timezone.utc))
    descricao         = Column(Text)

    __table_args__ = (
        CheckConstraint(
            "tipo_movimentacao IN ('debito','credito','estorno','ajuste')",
            name="ck_mov_tipo"
        ),
    )

    conta  = relationship("Conta_Pontos", back_populates="movimentacoes")
    aposta = relationship("Aposta",       back_populates="movimentacoes")

    def __repr__(self):
        return f"<Mov {self.tipo_movimentacao} | {self.pontos}pts>"


# ─────────────────────────────────────
# CRIAR TODAS AS TABELAS
# ─────────────────────────────────────
Base.metadata.create_all(engine)
print("✅ Todas as tabelas criadas com sucesso!")

2026-07-02 21:27:36,130 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-02 21:27:36,133 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("fase")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_info("fase")


2026-07-02 21:27:36,134 INFO sqlalchemy.engine.Engine [raw sql] ()


...

2026-07-02 21:27:36,300 INFO sqlalchemy.engine.Engine [no key 0.00134s] ()


INFO:sqlalchemy.engine.Engine:[no key 0.00134s] ()


2026-07-02 21:27:36,314 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


✅ Todas as tabelas criadas com sucesso!


In [10]:
# Célula 4 — Seed OFICIAL: Fases, Grupos e Seleções Copa 2026
from sqlalchemy.orm import Session

with Session(engine) as session:

    # ─────────────────────────────────────
    # FASES
    # ─────────────────────────────────────
    fases = [
    Fase(nome="Fase de Grupos",    ordem=1, tipo="grupos"),
    Fase(nome="Rodada de 32",      ordem=2, tipo="eliminatoria"),  # ← NOVA
    Fase(nome="Oitavas de Final",  ordem=3, tipo="eliminatoria"),
    Fase(nome="Quartas de Final",  ordem=4, tipo="eliminatoria"),
    Fase(nome="Semifinal",         ordem=5, tipo="eliminatoria"),
    Fase(nome="Disputa 3o Lugar",  ordem=6, tipo="eliminatoria"),
    Fase(nome="Final",             ordem=7, tipo="eliminatoria"),
    ]
    session.add_all(fases)
    session.flush()

    fase_grupos = fases[0]

    # ─────────────────────────────────────
    # GRUPOS A → L  (12 grupos Copa 2026)
    # ─────────────────────────────────────
    letras = ["A","B","C","D","E","F","G","H","I","J","K","L"]
    grupos = [
        Grupo(id_fase=fase_grupos.id_fase, nome=f"Grupo {l}")
        for l in letras
    ]
    session.add_all(grupos)
    session.flush()

    grupo_map = {g.nome.split()[-1]: g for g in grupos}

    # ─────────────────────────────────────
    # SELEÇÕES OFICIAIS Copa 2026
    # ─────────────────────────────────────
    selecoes_dados = {
        "A": [
            ("México",            "MEX"),
            ("África do Sul",     "RSA"),
            ("Coreia do Sul",     "KOR"),
            ("República Tcheca",  "CZE"),
        ],
        "B": [
            ("Suíça",   "SUI"),
            ("Canadá",  "CAN"),
            ("Bósnia",  "BIH"),
            ("Catar",   "QAT"),
        ],
        "C": [
            ("Brasil",   "BRA"),
            ("Marrocos", "MAR"),
            ("Escócia",  "SCO"),
            ("Haiti",    "HAI"),
        ],
        "D": [
            ("Estados Unidos", "USA"),
            ("Austrália",      "AUS"),
            ("Paraguai",       "PAR"),
            ("Turquia",        "TUR"),
        ],
        "E": [
            ("Alemanha",       "GER"),
            ("Curaçau",        "CUW"),
            ("Costa do Marfim","CIV"),
            ("Equador",        "ECU"),
        ],
        "F": [
            ("Holanda", "NED"),
            ("Japão",   "JPN"),
            ("Tunísia", "TUN"),
            ("Suécia",  "SWE"),
        ],
        "G": [
            ("Bélgica",       "BEL"),
            ("Egito",         "EGY"),
            ("Irã",           "IRN"),
            ("Nova Zelândia", "NZL"),
        ],
        "H": [
            ("Espanha",        "ESP"),
            ("Cabo Verde",     "CPV"),
            ("Arábia Saudita", "KSA"),
            ("Uruguai",        "URU"),
        ],
        "I": [
            ("França",   "FRA"),
            ("Senegal",  "SEN"),
            ("Iraque",   "IRQ"),
            ("Noruega",  "NOR"),
        ],
        "J": [
            ("Argentina", "ARG"),
            ("Argélia",   "ALG"),
            ("Áustria",   "AUT"),
            ("Jordânia",  "JOR"),
        ],
        "K": [
            ("Portugal",             "POR"),
            ("Rep. Dem. do Congo",   "COD"),
            ("Uzbequistão",          "UZB"),
            ("Colômbia",             "COL"),
        ],
        "L": [
            ("Inglaterra", "ENG"),
            ("Croácia",    "CRO"),
            ("Gana",       "GHA"),
            ("Panamá",     "PAN"),
        ],
    }

    todas_selecoes = []
    for letra, times in selecoes_dados.items():
        g = grupo_map[letra]
        for nome, sigla in times:
            todas_selecoes.append(
                Selecao(id_grupo=g.id_grupo, nome=nome, sigla=sigla)
            )

    session.add_all(todas_selecoes)
    session.commit()

    # ─────────────────────────────────────
    # RESUMO
    # ─────────────────────────────────────
    print(f"✅ {len(fases)} fases inseridas")
    print(f"✅ {len(grupos)} grupos inseridos (A → L)")
    print(f"✅ {len(todas_selecoes)} seleções inseridas")
    print()
    print("─── Times por grupo ───")
    for letra, times in selecoes_dados.items():
        nomes = " | ".join(t[0] for t in times)
        print(f"  Grupo {letra}: {nomes}")

2026-07-02 21:43:56,370 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-02 21:43:56,372 INFO sqlalchemy.engine.Engine INSERT INTO fase (nome, ordem, tipo) VALUES (?, ?, ?) RETURNING id_fase


INFO:sqlalchemy.engine.Engine:INSERT INTO fase (nome, ordem, tipo) VALUES (?, ?, ?) RETURNING id_fase


2026-07-02 21:43:56,375 INFO sqlalchemy.engine.Engine [cached since 376s ago (insertmanyvalues) 1/7 (ordered; batch not supported)] ('Fase de Grupos', 1, 'grupos')


...

2026-07-02 21:43:56,548 INFO sqlalchemy.engine.Engine [insertmanyvalues 48/48 (ordered; batch not supported)] (12, 'Panamá', 'PAN')


INFO:sqlalchemy.engine.Engine:[insertmanyvalues 48/48 (ordered; batch not supported)] (12, 'Panamá', 'PAN')


2026-07-02 21:43:56,550 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


✅ 7 fases inseridas
✅ 12 grupos inseridos (A → L)
✅ 48 seleções inseridas

─── Times por grupo ───
  Grupo A: México | África do Sul | Coreia do Sul | República Tcheca
  Grupo B: Suíça | Canadá | Bósnia | Catar
  Grupo C: Brasil | Marrocos | Escócia | Haiti
  Grupo D: Estados Unidos | Austrália | Paraguai | Turquia
  Grupo E: Alemanha | Curaçau | Costa do Marfim | Equador
  Grupo F: Holanda | Japão | Tunísia | Suécia
  Grupo G: Bélgica | Egito | Irã | Nova Zelândia
  Grupo H: Espanha | Cabo Verde | Arábia Saudita | Uruguai
  Grupo I: França | Senegal | Iraque | Noruega
  Grupo J: Argentina | Argélia | Áustria | Jordânia
  Grupo K: Portugal | Rep. Dem. do Congo | Uzbequistão | Colômbia
  Grupo L: Inglaterra | Croácia | Gana | Panamá


In [9]:
# Célula 4a — Limpar banco para reinserção
from sqlalchemy import text

with Session(engine) as session:
    session.execute(text("DELETE FROM movimentacao_pontos"))
    session.execute(text("DELETE FROM aposta"))
    session.execute(text("DELETE FROM conta_pontos"))
    session.execute(text("DELETE FROM usuario"))
    session.execute(text("DELETE FROM partida"))
    session.execute(text("DELETE FROM selecao"))
    session.execute(text("DELETE FROM grupo"))
    session.execute(text("DELETE FROM fase"))
    session.commit()
    print("✅ Banco limpo com sucesso")

2026-07-02 21:43:46,218 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-02 21:43:46,221 INFO sqlalchemy.engine.Engine DELETE FROM movimentacao_pontos


INFO:sqlalchemy.engine.Engine:DELETE FROM movimentacao_pontos


2026-07-02 21:43:46,222 INFO sqlalchemy.engine.Engine [generated in 0.00133s] ()


...

2026-07-02 21:43:46,243 INFO sqlalchemy.engine.Engine [generated in 0.00138s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00138s] ()


2026-07-02 21:43:46,244 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


✅ Banco limpo com sucesso
